In [1]:
import sys
import importlib
sys.path.insert(0, '/pscratch/sd/y/yuejian/envs/fairchemV2/lib/python3.10/site-packages')
sys.path.insert(0, '/global/homes/y/yuejian/project/MLFF-distill/APPLICATIONS/electrolytes/plots/MD_vis')

from ase.io import Trajectory
import vis
importlib.reload(vis)
from vis import view_frame, animate_frames, browse_frames, save_gif

In [2]:
TRAJ_PATH = (
    "/global/homes/y/yuejian/project/MLFF-distill/m5024/distillation_project/data/raw_data_from_UMA_simulation/413K/0.1M/md_omol_napf6_dme_re1/md_omol_napf6_dme_re1.traj"
)

# Open lazily — no frames are loaded into memory yet
traj = Trajectory(TRAJ_PATH)
print(f'Trajectory opened: {len(traj)} frames')

Trajectory opened: 20002 frames


In [3]:
# # --- Animation parameters ---
# ANIM_START  = 0
# ANIM_END    = 200000
# ANIM_STRIDE = 5000
# ANIM_STYLE  = 'sphere'   # 'sphere' | 'ball+stick'
# ANIM_HIDE_H = True

# v = animate_frames(traj,
#                    start=ANIM_START, end=ANIM_END, stride=ANIM_STRIDE,
#                    style=ANIM_STYLE, hide_H=ANIM_HIDE_H)

In [4]:
# FRAME_IDX = 0
# SNAP_STYLE  = 'sphere'   # 'sphere' | 'stick' | 'ball+stick'
# SNAP_HIDE_H = False

# v = view_frame(traj, frame_idx=FRAME_IDX, style=SNAP_STYLE, hide_H=SNAP_HIDE_H)

In [5]:
# GIF_OUTPUT_DIR = '/global/homes/y/yuejian/project/MLFF-distill/yuejian/electrolyte_application/gif'
# GIF_START  = 0
# GIF_END    = 99
# GIF_STRIDE = 10    # increase to reduce frames → smaller file and faster
# GIF_HIDE_H = True
# GIF_FPS    = 5
# GIF_SIZE   = 400   # pixel width/height — smaller = faster and smaller file
# GIF_NAME   = os.path.basename(traj.filename).replace('.traj', '.gif')

# import os
# save_gif(traj,
#          output_path=os.path.join(GIF_OUTPUT_DIR, GIF_NAME),
#          start=GIF_START, end=GIF_END, stride=GIF_STRIDE,
#          hide_H=GIF_HIDE_H, fps=GIF_FPS, size=GIF_SIZE)

In [7]:
from vis import save_gif_3d
import os, re

KNOWN_CATIONS  = ['na', 'li', 'k', 'mg', 'ca']
KNOWN_ANIONS   = ['pf6', 'tfsi', 'fsi', 'bf4', 'otf', 'dca', 'no3']
KNOWN_SOLVENTS = ['diglyme', 'dme', 'ec', 'dmc', 'pc', 'thf', 'acn', 'dmso', 'water']

_CONC_RE = re.compile(r'(\d+(?:_\d+)?)M')   # matches 1M, 0_1M, 2_5M …

def parse_traj_name(traj_path):
    parts = traj_path.replace('\\', '/').split('/')

    # Temperature: directory matching e.g. 298K
    temperature = next((p for p in parts if re.fullmatch(r'\d+K', p)), 'unknownK')

    # Concentration: \d+M or \d+_\d+M (underscore = decimal point)
    conc_part     = next((p for p in parts if _CONC_RE.search(p)), '')
    conc_raw      = _CONC_RE.search(conc_part).group(1) if conc_part else None
    concentration = (conc_raw.replace('_', '.') + 'M') if conc_raw else 'unknownM'

    # Model type from path keywords
    if   any('micro'  in p for p in parts): model = 'micro'
    elif any('origin' in p or 'orig' in p for p in parts): model = 'original'
    else: model = 'unknown'

    # System name from the run directory (deepest dir before the .traj file)
    run_dir = os.path.basename(os.path.dirname(traj_path)).lower()
    cation  = next((c for c in KNOWN_CATIONS  if c in run_dir), 'unknown')
    anion   = next((a for a in KNOWN_ANIONS   if a in run_dir), 'unknown')
    solvent = next((s for s in KNOWN_SOLVENTS if s in run_dir), 'unknown')

    return temperature, concentration, model, cation, anion, solvent

temperature, concentration, model, cation, anion, solvent = parse_traj_name(traj.filename)
print(f'Parsed: model={model} | {cation.upper()}/{anion.upper()} in {solvent} | {concentration} | {temperature}')

GIF3D_OUTPUT_DIR = '/global/homes/y/yuejian/project/MLFF-distill/yuejian/electrolyte_application/gif'
GIF3D_START      = 0
GIF3D_END        = 20000 # 200000
GIF3D_STRIDE     = 100
GIF3D_HIDE_H     = True
GIF3D_FPS        = 5
GIF3D_SIZE       = 400
GIF3D_ELEV       = 20
GIF3D_AZIM       = 45
GIF3D_SPIN       = False
GIF3D_WORKERS    = 4
GIF3D_ATOM_SCALE = 0.5
# Interval between saved frames in fs (not the MD integration timestep).
# 20 ns run / 200 000 frames = 100 fs per frame. Displayed as ns on each frame.
# Set to 0 to disable the time stamp.
GIF3D_TIMESTEP_FS = 10
GIF3D_NAME        = f'{model}_{cation}{anion}_{solvent}_{concentration}_{temperature}_3d.gif'

print(f'Output: {GIF3D_NAME}')
save_gif_3d(traj,
            output_path=os.path.join(GIF3D_OUTPUT_DIR, GIF3D_NAME),
            start=GIF3D_START, end=GIF3D_END, stride=GIF3D_STRIDE,
            hide_H=GIF3D_HIDE_H, fps=GIF3D_FPS, size=GIF3D_SIZE,
            elev=GIF3D_ELEV, azim=GIF3D_AZIM, spin=GIF3D_SPIN,
            n_workers=GIF3D_WORKERS, atom_scale=GIF3D_ATOM_SCALE,
            timestep_fs=GIF3D_TIMESTEP_FS,
            label=os.path.splitext(GIF3D_NAME)[0])

Parsed: model=unknown | NA/PF6 in dme | 1M | 413K
Output: unknown_napf6_dme_1M_413K_3d.gif
3D GIF saved → /global/homes/y/yuejian/project/MLFF-distill/yuejian/electrolyte_application/gif/unknown_napf6_dme_1M_413K_3d.gif  (201 frames, 5 fps, 1.67 MB)
